# Sprint 2 optional implementation companion

Follow the README's direct commands and use the actual MLflow UI first. This notebook shows native MLflow/scikit-learn operations when you want to inspect the evidence more closely. It uses your own local run IDs from `work/comparison.json`, not the authored reference IDs. Complete the tracked study and comparison before executing it. Keep independent attempts before opening feedback.

In [ ]:
from pathlib import Path
import os, csv, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import mlflow
from mlflow import MlflowClient
ROOT = Path.cwd()
if not (ROOT / "src/catalog_mlops").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
assert (ROOT / "work/comparison.json").is_file(), "Complete the tracked study and raw comparison first"
report = json.loads((ROOT / "work/comparison.json").read_text())
uri = "sqlite:///" + (ROOT / "work/mlflow/tracking.db").resolve().as_posix()
client = MlflowClient(tracking_uri=uri)
mlflow.set_tracking_uri(uri)
print("PyTorch", torch.__version__, "MLflow", mlflow.__version__)


## Why ad hoc tracking breaks down at scale

Read one native MLflow run object. Identify its recipe parameters, final metric and tags. A recorded object makes these observations inspectable; it does not prove that two runs are comparable.

In [ ]:
baseline = client.get_run(report["baseline_run_id"])
print(baseline.info.run_id, baseline.info.status)
print(baseline.data.params)
print("Final macro F1", baseline.data.metrics["validation_macro_f1"])
print({key: baseline.data.tags[key] for key in ["source_sha256", "split_sha256", "data_sha256"]})


## Controls, confounders, and decision metrics

Inspect the comparison controls and distinguish the primary metric from category/runtime diagnostics. The comparison file was already checked by the project verifier; read its source to understand those checks.

In [ ]:
study = [row for row in report["rows"] if row.get("comparable") and row["kind"] in {"baseline", "candidate"}]
assert len(study) >= 6
for row in study:
    print(row["name"], row["changed_field"], row["validation_macro_f1"])
for row in study[1:]:
    assert row["controls"] == study[0]["controls"]
print("Fixed controls", study[0]["controls"])


## Building the five-run experiment matrix

Keep your own five hypotheses and configuration differences in the project plan. Do not replace that reasoning with the printed reference configurations.

In [ ]:
# Record your five one-factor hypotheses and one excluded/confounded proposal after making your plan.

<details><summary>Reference feedback</summary>

**Result you should get**

Five distinct candidates change one training field each, against the same baseline and evaluation conditions. A repeat or two-factor row does not count as another one-factor candidate.

**How**

Compare each configuration with baseline, not with the previous candidate. Keep source/data/environment/initialization controls fixed and choose macro F1 before seeing results.

</details>

## Revisiting MLflow in a pipeline context

Inspect parameters, metrics and a downloaded artifact using the real MLflow client. The CSV is the observed prediction evidence, not a prepared answer table.

In [ ]:
download = ROOT / "work/notebook-artifacts"
download.mkdir(exist_ok=True)
raw_checks = []
class_names = ("T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot")
for row in study:
    destination = download / row["run_id"]
    destination.mkdir(exist_ok=True)
    csv_path = client.download_artifacts(row["run_id"], "recipe/validation-predictions.csv", str(destination))
    with open(csv_path) as stream:
        predictions = list(csv.DictReader(stream))
    truth = np.array([int(item["label"]) for item in predictions])
    predicted = np.array([int(item["prediction"]) for item in predictions])
    score = f1_score(truth, predicted, average="macro")
    assert abs(score - client.get_run(row["run_id"]).data.metrics["validation_macro_f1"]) < 1e-12
    raw_checks.append({"name":row["name"], "run_id":row["run_id"], "macro_f1":score,
                       "accuracy":accuracy_score(truth,predicted),
                       "classes":classification_report(truth,predicted,labels=range(10),target_names=class_names,output_dict=True,zero_division=0)})
print("Recomputed", len(raw_checks), "raw prediction reports")


## Comparing multiple runs systematically

Use the real UI comparison before checking this plot. The bars show raw CSV recomputations and use the full 0–1 metric scale. Keep your own selection and category tradeoffs.

In [ ]:
figure, axis = plt.subplots(figsize=(10, 4.5), layout="constrained")
axis.bar([row["name"] for row in raw_checks], [row["macro_f1"] for row in raw_checks], color="#337F57")
axis.set(ylabel="Validation macro F1 (unitless)", ylim=(0,1), xlabel="Configuration; same 3,000 validation images")
axis.tick_params(axis="x", rotation=25)
plt.show()


In [ ]:
# Write your selection, exact raw gain and at least one category regression before opening feedback.

<details><summary>Reference feedback</summary>

**Result you should get**

Choose the largest eligible final macro F1, preserve its full run/model IDs and report regressions. The authored study selected eight epochs; another result must be judged from its own fixed controls and raw values.

**How**

Compare full-precision values, apply the declared tie rule and inspect category precision/recall. A better average does not imply every class improved.

</details>

## Autologging and reducing manual tracking overhead

Read the actual TensorBoard-hook histories directly. The scalar event representation is float32; final explicitly logged decision metrics retain their original precision.

In [ ]:
receipt = json.loads((ROOT / "work/experiments/baseline-autolog.tracking.json").read_text())
auto_id = receipt["run_id"]
automatic = client.get_run(auto_id)
assert automatic.data.params["autolog_hook"] == "tensorboard"
history_path = client.download_artifacts(auto_id, "recipe/history.json", str(download))
observations = json.loads(Path(history_path).read_text())
for field in ("train_loss", "validation_loss", "validation_accuracy", "validation_macro_f1"):
    history = sorted(client.get_metric_history(auto_id, "auto/"+field), key=lambda item:item.step)
    assert [item.step for item in history] == [row["epoch"] for row in observations]
    np.testing.assert_allclose([item.value for item in history], [np.float32(row[field]) for row in observations], rtol=0, atol=1e-12)
    print(field, [(item.step,item.value) for item in history])
assert automatic.data.metrics["validation_macro_f1"] == baseline.data.metrics["validation_macro_f1"]


## Validating an assistant-generated experiment summary

Retain your actual assistant request, response and claim decisions outside the notebook. Compare every recommendation against native records and raw predictions; do not substitute the authored demonstration for your request.

In [ ]:
# Record one accepted, qualified or rejected assistant claim with its exact supporting run ID and field.

<details><summary>Reference feedback</summary>

**Result you should get**

Each recommendation has a traceable ID, primary metric, valid conditions and stated limits. Constructed false claims remain separate from the actual response.

**How**

Check parameters/tags, raw final metrics, category results and model attribution. Correct unsupported claims while preserving the original assistant response.

</details>

## Experiment tracking deliverable

Apply Morrow Market’s predeclared recall requirements before maximizing macro F1. Keep a distinct partner decision record; the original experimental ranking remains intact.

In [ ]:
# Apply the partner eligibility checks and justify the selected eligible configuration using your own raw values.

<details><summary>Reference feedback</summary>

**Result you should get**

Require T-shirt/top recall at least 0.80 and sneaker recall at least 0.90, then maximize macro F1 among comparable eligible configurations. The author study selected weight decay; a no-eligible-result conclusion is valid when supported by your observations.

**How**

Separate control validity from business acceptance. Validate the partner assistant response against both thresholds and preserve any corrections.

</details>

## Summary

Retain actual run IDs, a controlled comparison, automatic metric-history evidence and checked recommendations. The model selected for further validation must remain connected to its evaluated weights and recipe.